# Préparation des données : fusion et feature engineering
Adapté du kernel Kaggle « LightGBM with Simple Features » (jsaguiar).
Adaptations : compatibilité pandas récent, correction de l'anomalie DAYS_EMPLOYED avec indicateur,
features métier du notebook 01, contrôle des lignes après chaque jointure, nettoyage des noms de colonnes.

In [1]:
import gc, re
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

def one_hot_encoder(df, nan_as_category=True):
    """Transforme chaque colonne texte en colonnes 0/1 (une par catégorie)."""
    original = list(df.columns)
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    df = pd.get_dummies(df, columns=cat_cols, dummy_na=nan_as_category, dtype=int)
    new_cols = [c for c in df.columns if c not in original]
    return df, new_cols

In [2]:
def application_train_test(nan_as_category=False):
    train = pd.read_csv(DATA / "application_train.csv")
    test = pd.read_csv(DATA / "application_test.csv")
    print(f"Train : {len(train)} | Test : {len(test)}")
    df = pd.concat([train, test], ignore_index=True)   # même encodage pour les deux
    del train, test; gc.collect()

    # 4 clients au genre "XNA" (non renseigné) : supprimés
    df = df[df["CODE_GENDER"] != "XNA"]

    # Variables à 2 valeurs -> 0/1
    for col in ["CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY"]:
        df[col], _ = pd.factorize(df[col])
    df, _ = one_hot_encoder(df, nan_as_category)

    # Anomalie DAYS_EMPLOYED (notebook 01)
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    # Features métier (notebook 01) + revenu par personne (kernel)
    df["CREDIT_INCOME_RATIO"]  = df["AMT_CREDIT"]  / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["PAYMENT_RATE"]         = df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    df["EMPLOYED_BIRTH_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["EXT_SOURCES_MEAN"]     = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
    df["INCOME_PER_PERSON"]    = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    return df

df = application_train_test()
print(df.shape)
print(f"Lignes train : {df['TARGET'].notna().sum()} | test : {df['TARGET'].isna().sum()}")

Train : 307511 | Test : 48744
(356251, 249)
Lignes train : 307507 | test : 48744


C:\Users\A491274\AppData\Local\Temp\ipykernel_38860\3907086279.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
C:\Users\A491274\AppData\Local\Temp\ipykernel_38860\3907086279.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["CREDIT_INCOME_RATIO"]  = df["AMT_CREDIT"]  / df["AMT_INCOME_TOTAL"]
C:\Users\A491274\AppData\Local\Temp\ipykernel_38860\3907086279.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

In [3]:
def bureau_and_balance(nan_as_category=True):
    bureau = pd.read_csv(DATA / "bureau.csv")
    bb = pd.read_csv(DATA / "bureau_balance.csv")
    bb, bb_cat = one_hot_encoder(bb, nan_as_category)
    bureau, bureau_cat = one_hot_encoder(bureau, nan_as_category)

    # Niveau 1 : historique mensuel -> 1 ligne par crédit
    bb_aggs = {"MONTHS_BALANCE": ["min", "max", "size"]}
    for col in bb_cat:
        bb_aggs[col] = ["mean"]
    bb_agg = bb.groupby("SK_ID_BUREAU").agg(bb_aggs)
    bb_agg.columns = [f"{c}_{s.upper()}" for c, s in bb_agg.columns]
    bureau = bureau.join(bb_agg, how="left", on="SK_ID_BUREAU").drop(columns="SK_ID_BUREAU")
    del bb, bb_agg; gc.collect()

    # Niveau 2 : crédits -> 1 ligne par client
    num_aggs = {
        "DAYS_CREDIT": ["min", "max", "mean", "var"],
        "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"],
        "CREDIT_DAY_OVERDUE": ["max", "mean"],
        "AMT_CREDIT_MAX_OVERDUE": ["mean"],
        "AMT_CREDIT_SUM": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["mean"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "sum"],
        "AMT_ANNUITY": ["max", "mean"],
        "CNT_CREDIT_PROLONG": ["sum"],
        "MONTHS_BALANCE_MIN": ["min"],
        "MONTHS_BALANCE_MAX": ["max"],
        "MONTHS_BALANCE_SIZE": ["mean", "sum"],
    }
    cat_aggs = {c: ["mean"] for c in bureau_cat}
    cat_aggs.update({f"{c}_MEAN": ["mean"] for c in bb_cat})

    agg = bureau.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    agg.columns = [f"BURO_{c}_{s.upper()}" for c, s in agg.columns]

    # Même résumé, séparément pour les crédits en cours et les crédits clôturés
    for statut, prefixe in [("Active", "ACTIVE"), ("Closed", "CLOSED")]:
        sous = bureau[bureau[f"CREDIT_ACTIVE_{statut}"] == 1]
        sous_agg = sous.groupby("SK_ID_CURR").agg(num_aggs)
        sous_agg.columns = [f"{prefixe}_{c}_{s.upper()}" for c, s in sous_agg.columns]
        agg = agg.join(sous_agg, how="left")

    del bureau; gc.collect()
    return agg

bureau = bureau_and_balance()
print(f"bureau agrégé : {bureau.shape}")

n_avant = len(df)
df = df.join(bureau, how="left", on="SK_ID_CURR")
assert len(df) == n_avant, "La jointure a changé le nombre de lignes !"
del bureau; gc.collect()

print(f"Après jointure : {df.shape}")
print(f"Clients SANS aucun crédit ailleurs : {df['BURO_DAYS_CREDIT_MEAN'].isna().mean():.1%}")

bureau agrégé : (305811, 116)
Après jointure : (356251, 365)
Clients SANS aucun crédit ailleurs : 14.2%


- application : 307 507 clients train (4 XNA retirés) + 48 744 test ; 249 colonnes après encodage.
- bureau + bureau_balance : 116 variables agrégées ; jointure gauche sans perte de lignes.
- 14,2 % des clients n'ont aucun crédit dans un autre organisme (population cible de l'entreprise).

In [4]:
def joindre(df, agg, nom):
    n_avant = len(df)
    df = df.join(agg, how="left", on="SK_ID_CURR")
    assert len(df) == n_avant, f"{nom} : la jointure a changé le nombre de lignes !"
    print(f"{nom:<14} {agg.shape[0]:>7} clients | +{agg.shape[1]:>3} colonnes → df {df.shape}")
    return df

In [5]:
def previous_applications(nan_as_category=True):
    prev = pd.read_csv(DATA / "previous_application.csv")
    prev, cat_cols = one_hot_encoder(prev, nan_as_category)

    # Même code aberrant 365243 que DAYS_EMPLOYED (écriture corrigée, cf. piège inplace)
    for col in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
                "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
        prev[col] = prev[col].replace(365243, np.nan)

    # Montant demandé / montant accordé
    prev["APP_CREDIT_PERC"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]

    num_aggs = {
        "AMT_ANNUITY": ["min", "max", "mean"],
        "AMT_APPLICATION": ["min", "max", "mean"],
        "AMT_CREDIT": ["min", "max", "mean"],
        "APP_CREDIT_PERC": ["min", "max", "mean", "var"],
        "AMT_DOWN_PAYMENT": ["min", "max", "mean"],
        "AMT_GOODS_PRICE": ["min", "max", "mean"],
        "HOUR_APPR_PROCESS_START": ["min", "max", "mean"],
        "RATE_DOWN_PAYMENT": ["min", "max", "mean"],
        "DAYS_DECISION": ["min", "max", "mean"],
        "CNT_PAYMENT": ["mean", "sum"],
    }
    cat_aggs = {c: ["mean"] for c in cat_cols}

    agg = prev.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    agg.columns = [f"PREV_{c}_{s.upper()}" for c, s in agg.columns]

    # Même résumé, séparément pour les demandes acceptées et refusées
    for statut, prefixe in [("Approved", "APPROVED"), ("Refused", "REFUSED")]:
        sous = prev[prev[f"NAME_CONTRACT_STATUS_{statut}"] == 1]
        sous_agg = sous.groupby("SK_ID_CURR").agg(num_aggs)
        sous_agg.columns = [f"{prefixe}_{c}_{s.upper()}" for c, s in sous_agg.columns]
        agg = agg.join(sous_agg, how="left")

    del prev; gc.collect()
    return agg

prev = previous_applications()
df = joindre(df, prev, "previous_app")
del prev; gc.collect()

previous_app    338857 clients | +249 colonnes → df (356251, 614)


0

In [6]:
def pos_cash(nan_as_category=True):
    pos = pd.read_csv(DATA / "POS_CASH_balance.csv")
    pos, cat_cols = one_hot_encoder(pos, nan_as_category)
    aggs = {
        "MONTHS_BALANCE": ["max", "mean", "size"],
        "SK_DPD": ["max", "mean"],          # jours de retard
        "SK_DPD_DEF": ["max", "mean"],
    }
    aggs.update({c: ["mean"] for c in cat_cols})
    agg = pos.groupby("SK_ID_CURR").agg(aggs)
    agg.columns = [f"POS_{c}_{s.upper()}" for c, s in agg.columns]
    agg["POS_COUNT"] = pos.groupby("SK_ID_CURR").size()
    del pos; gc.collect()
    return agg


def installments_payments(nan_as_category=True):
    ins = pd.read_csv(DATA / "installments_payments.csv")
    ins, cat_cols = one_hot_encoder(ins, nan_as_category)
    # Part payée et écart pour chaque échéance
    ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    # Jours de retard (DPD) et d'avance (DBD), bornés à 0 : clip remplace le apply lent du kernel
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)
    aggs = {
        "NUM_INSTALMENT_VERSION": ["nunique"],
        "DPD": ["max", "mean", "sum"],
        "DBD": ["max", "mean", "sum"],
        "PAYMENT_PERC": ["max", "mean", "sum", "var"],
        "PAYMENT_DIFF": ["max", "mean", "sum", "var"],
        "AMT_INSTALMENT": ["max", "mean", "sum"],
        "AMT_PAYMENT": ["min", "max", "mean", "sum"],
        "DAYS_ENTRY_PAYMENT": ["max", "mean", "sum"],
    }
    aggs.update({c: ["mean"] for c in cat_cols})
    agg = ins.groupby("SK_ID_CURR").agg(aggs)
    agg.columns = [f"INSTAL_{c}_{s.upper()}" for c, s in agg.columns]
    agg["INSTAL_COUNT"] = ins.groupby("SK_ID_CURR").size()
    del ins; gc.collect()
    return agg


def credit_card_balance(nan_as_category=True):
    cc = pd.read_csv(DATA / "credit_card_balance.csv")
    cc, _ = one_hot_encoder(cc, nan_as_category)
    cc = cc.drop(columns="SK_ID_PREV")
    agg = cc.groupby("SK_ID_CURR").agg(["min", "max", "mean", "sum", "var"])
    agg.columns = [f"CC_{c}_{s.upper()}" for c, s in agg.columns]
    agg["CC_COUNT"] = cc.groupby("SK_ID_CURR").size()
    del cc; gc.collect()
    return agg


for nom, fonction in [("pos_cash", pos_cash),
                      ("installments", installments_payments),
                      ("credit_card", credit_card_balance)]:
    agg = fonction()
    df = joindre(df, agg, nom)
    del agg; gc.collect()

pos_cash        337252 clients | + 18 colonnes → df (356251, 632)
installments    339587 clients | + 26 colonnes → df (356251, 658)
credit_card     103558 clients | +141 colonnes → df (356251, 799)


In [7]:
# 1) Divisions par zéro -> infini -> remplacé par vide
n_inf = np.isinf(df.select_dtypes("number")).sum().sum()
df = df.replace([np.inf, -np.inf], np.nan)
print(f"Valeurs infinies remplacées : {n_inf}")

# 2) Noms de colonnes : LightGBM refuse les caractères spéciaux (espaces, virgules, ':'...)
df.columns = [re.sub(r"[^A-Za-z0-9_]", "_", c) for c in df.columns]
assert df.columns.is_unique, "Deux colonnes ont le même nom après nettoyage !"

# 3) float64 -> float32 : moitié moins de mémoire, précision largement suffisante
cols_float = df.select_dtypes("float64").columns
df[cols_float] = df[cols_float].astype("float32")

# 4) Séparation train / test et sauvegarde
train_df = df[df["TARGET"].notna()].reset_index(drop=True)
test_df = df[df["TARGET"].isna()].drop(columns="TARGET").reset_index(drop=True)

train_df.to_parquet(OUT / "train_final.parquet")
test_df.to_parquet(OUT / "test_final.parquet")

print(f"Train final : {train_df.shape} | Test final : {test_df.shape}")
print(f"Mémoire train : {train_df.memory_usage().sum() / 1e6:.0f} Mo")
print(f"Clients sans ancienne demande Home Credit : {train_df['PREV_AMT_CREDIT_MEAN'].isna().mean():.1%}")

Valeurs infinies remplacées : 61
Train final : (307507, 799) | Test final : (48744, 798)
Mémoire train : 1199 Mo
Clients sans ancienne demande Home Credit : 5.4%


### Résultat de la préparation
- 4 tables Home Credit agrégées par client (préfixes PREV_, APPROVED_, REFUSED_, POS_, INSTAL_, CC_) ; aucune jointure n'a modifié le nombre de lignes.
- Seuls ~29 % des clients ont une carte de crédit : les variables CC_ sont majoritairement vides (information conservée).
- 61 valeurs infinies (divisions par zéro) remplacées par NaN ; noms de colonnes nettoyés pour LightGBM ; float64 → float32.
- Dataset final : 307 507 clients × 797 features (+ ID + TARGET), sauvegardé en Parquet.